In [1]:
import pandas as pd, json

sheets = pd.read_excel("Supply chain logisitcs problem.xlsx", sheet_name=None)
for df in sheets.values():
    df.columns = [c.strip() for c in df.columns]

orders = sheets["OrderList"]
freight = sheets["FreightRates"].drop_duplicates()
plant_ports = sheets["PlantPorts"]
products = sheets["ProductsPerPlant"]

nodes, links = [], []

ports = (set(orders["Origin Port"]) | set(orders["Destination Port"])
         | set(freight["orig_port_cd"]) | set(freight["dest_port_cd"])
         | set(plant_ports["Port"]))
plants = (set(plant_ports["Plant Code"]) | set(products["Plant Code"])
          | set(orders["Plant Code"]))

for p in ports:
    nodes.append({"id": p, "type": "Port"})
for p in plants:
    nodes.append({"id": p, "type": "Plant"})
for c in set(orders["Customer"]):
    nodes.append({"id": c, "type": "Customer"})
for p in set(products["Product ID"].astype(str)):
    nodes.append({"id": p, "type": "Product"})

for _, r in plant_ports.iterrows():
    links.append({"source": r["Plant Code"], "target": r["Port"], "type": "HOSTED_AT"})

routes = freight[["orig_port_cd", "dest_port_cd"]].drop_duplicates()
for _, r in routes.iterrows():
    links.append({"source": r["orig_port_cd"], "target": r["dest_port_cd"], "type": "ROUTE"})

for _, r in products.iterrows():
    links.append({"source": r["Plant Code"], "target": str(r["Product ID"]), "type": "MAKES"})

pairs = orders.groupby(["Plant Code", "Customer"]).agg(
    orders=("Order ID", "count"),
    late=("Ship Late Day count", "mean")).reset_index()
for _, r in pairs.iterrows():
    links.append({"source": r["Plant Code"], "target": r["Customer"],
                  "type": "ORDERED_BY", "orders": int(r["orders"]),
                  "late_days": round(float(r["late"]), 2)})

with open("graph.json", "w") as f:
    json.dump({"nodes": nodes, "links": links}, f)
print(len(nodes), "nodes,", len(links), "links")

1617 nodes, 2141 links


In [2]:
import shutil, os
shutil.copy("graph.json", os.path.expanduser("~/atmograph-ui/public/graph.json"))
print("copied")

copied
